In [ ]:
import numpy as np

def sigmoid(x):
    """
    Sigmoid activation function
    :param x: input value (float or np.array)
    :return: sigmoid of x
    """
    return 1.0 / (1.0 + np.exp(-x))

def dsigmoid(x):
    """
    Derivative of the sigmoid function
    :param x: input value (float or np.array); x is the output of sigmoid
    :return: derivative of sigmoid at x
    """
    return x * (1.0 - x)

def tanh(x):
    """
    Hyperbolic tangent activation function
    :param x: input value (float or np.array)
    :return: tanh of x
    """
    return np.tanh(x)

def dtanh(x):
    """
    Derivative of the hyperbolic tangent function
    :param x: input value (float or np.array); x is the output of tanh
    :return: derivative of tanh at x
    """
    return 1.0 - x**2

class NeuralNetwork:
    """
    A simple fully-connected neural network implementation.
    Allows multiple hidden layers with a specified number of neurons each.
    Provides options for activation function: sigmoid or tanh.
    Includes bias neurons.
    """
    def __init__(self, input_size, hidden_layers, hidden_neurons, output_size,
                 activation='sigmoid', learning_rate=0.1, epochs=10000):
        """
        Initialize the neural network.

        :param input_size: number of input neurons (excluding bias)
        :param hidden_layers: integer, how many hidden layers
        :param hidden_neurons: list of integers specifying the number of neurons in each hidden layer
        :param output_size: number of output neurons
        :param activation: 'sigmoid' or 'tanh'
        :param learning_rate: learning rate (float)
        :param epochs: number of training epochs (integer)
        """
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.hidden_neurons = hidden_neurons
        self.output_size = output_size
        self.learning_rate = learning_rate
        self.epochs = epochs

        # Select activation function
        if activation == 'sigmoid':
            self.activation = sigmoid
            self.activation_deriv = dsigmoid
        else:
            self.activation = tanh
            self.activation_deriv = dtanh

        # Create a list of layer sizes: input -> hidden layers -> output
        layer_sizes = [self.input_size] + self.hidden_neurons + [self.output_size]

        # Initialize weights (with bias) for each layer:
        # Example: if layer_sizes = [2, 4, 1], we have:
        #   - weights[0] is shape (4, 3) for connecting input layer(2) + bias(1) -> hidden layer(4)
        #   - weights[1] is shape (1, 5) for connecting hidden layer(4) + bias(1) -> output layer(1)

        self.weights = []
        for i in range(len(layer_sizes)-1):
            w = np.random.uniform(-0.05, 0.05, (layer_sizes[i+1], layer_sizes[i] + 1))
            self.weights.append(w)

    def forward(self, X):
        """
        Forward pass through the network.
        :param X: input array of shape (n_features, ) for a single sample
        :return: (list_of_z, list_of_a) containing pre-activations(z) and activations(a) for each layer
        """
        # Convert X to a column vector if needed
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)

        z_values = []  # pre-activation values
        a_values = []  # activation values

        # The activation of the input layer is just X (flattened) but we add bias = 1
        a = np.vstack((X, [[1]]))  # shape (input_size+1, 1)
        a_values.append(a)

        # Forward pass through each layer
        for w in self.weights:
            z = np.dot(w, a)  # shape = (layer_size, 1)
            a_no_bias = self.activation(z)  # shape = (layer_size, 1)
            z_values.append(z)
            # Add bias to the activation except for the output layer
            if w is not self.weights[-1]:
                a = np.vstack((a_no_bias, [[1]]))
            else:
                a = a_no_bias
            a_values.append(a)

        return z_values, a_values

    def backward(self, z_values, a_values, y):
        """
        Backward pass (backpropagation).
        :param z_values: list of pre-activation values for each layer
        :param a_values: list of activation values for each layer
        :param y: target value (scalar or vector)
        :return: list of gradients for each weight matrix
        """
        # Convert y to the same shape as output layer if it's scalar
        if isinstance(y, (int, float)):
            y = np.array([[y]])
        else:
            y = y.reshape(-1,1)

        # Initialize list of deltas
        deltas = [None] * len(self.weights)

        # The activation at the output layer is the last in a_values
        a_output = a_values[-1]  # shape (output_size, 1)

        # 1) Output layer delta
        error = (a_output - y)  # shape (output_size, 1)
        if len(z_values) > 0:
            # derivative w.r.t. z_output
            delta_output = error * self.activation_deriv(a_output)
        else:
            delta_output = error  # in case there's no hidden layer
        deltas[-1] = delta_output

        # 2) Hidden layers delta (if any)
        # We go backwards from second to last layer to the first layer
        for i in range(len(self.weights) - 2, -1, -1):
            w_next = self.weights[i+1]
            # remove the bias row from next layer weights if needed
            w_next_no_bias = w_next[:, :-1]  # shape: (layer_{i+1}, layer_i)
            delta_up = deltas[i+1]  # shape: (layer_{i+1}, 1)

            # activation of layer i (a_values[i+1] includes bias if not the output)
            a_i = a_values[i+1]
            if i < len(self.weights) - 2:
                # if not the last hidden layer, then a_i includes the bias as last row
                a_i_no_bias = a_i[:-1]
            else:
                # for the hidden layer just before the output, a_i has no extra bias row if there's only one hidden layer
                if a_i.shape[0] > w_next_no_bias.shape[1]:
                    a_i_no_bias = a_i[:-1]
                else:
                    a_i_no_bias = a_i

            # we need the derivative of a_i (which is activation), so use activation_deriv
            delta_i = (w_next_no_bias.T @ delta_up) * self.activation_deriv(a_i_no_bias)
            deltas[i] = delta_i

        # Now compute gradients for each layer's weights
        grads = []
        for i, w in enumerate(self.weights):
            # Activation of previous layer
            a_prev = a_values[i]  # shape: (layer_size + 1, 1)
            # delta for current layer
            delta_i = deltas[i]  # shape: (layer_size, 1)
            # gradient is delta_i * a_prev^T
            grad_i = delta_i @ a_prev.T  # shape: same as w
            grads.append(grad_i)

        return grads

    def update_weights(self, grads):
        """
        Update weights by gradient descent step.
        :param grads: list of gradients for each layer
        """
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * grads[i]

    def fit(self, X, y):
        """
        Train (fit) the neural network on the given data.
        :param X: input data, shape (n_samples, n_features)
        :param y: target data, shape (n_samples,) or (n_samples, 1)
        """
        for epoch in range(self.epochs):
            # We'll do batch gradient descent for simplicity on these tiny datasets.
            total_grad = [np.zeros_like(w) for w in self.weights]

            for i in range(len(X)):
                x_sample = X[i]
                y_sample = y[i]
                z_values, a_values = self.forward(x_sample)
                grads = self.backward(z_values, a_values, y_sample)
                # accumulate
                for j in range(len(total_grad)):
                    total_grad[j] += grads[j]

            # After summing gradients over the entire dataset, update
            for j in range(len(total_grad)):
                total_grad[j] /= len(X)

            self.update_weights(total_grad)

    def predict(self, X):
        """
        Predict outputs for given input X.
        :param X: input array of shape (n_samples, n_features)
        :return: predictions array of shape (n_samples, )
        """
        preds = []
        for i in range(len(X)):
            _, a_values = self.forward(X[i])
            preds.append(a_values[-1].flatten())  # last layer's activation
        return np.array(preds).squeeze()

def get_boolean_data(func_name):
    """
    Return the input-output pairs for a given boolean function with 2 inputs.
    :param func_name: 'AND', 'OR', 'XOR'
    :return: (X, y) where X.shape = (4,2), y.shape=(4,)
    """
    X = np.array([
        [0, 0],
        [0, 1],
        [1, 0],
        [1, 1]
    ])

    if func_name == 'AND':
        y = np.array([0, 0, 0, 1])
    elif func_name == 'OR':
        y = np.array([0, 1, 1, 1])
    elif func_name == 'XOR':
        y = np.array([0, 1, 1, 0])
    else:
        raise ValueError("Unknown boolean function: "+func_name)

    return X, y

def train_and_test_boolean_function(func_name, activation, hidden_layers, hidden_neurons,
                                    learning_rate=0.1, epochs=10000):
    """
    Train a neural network to learn a boolean function and return the predictions.
    :param func_name: 'AND', 'OR', or 'XOR'
    :param activation: 0 for sigmoid, 1 for tanh
    :param hidden_layers: number of hidden layers (int)
    :param hidden_neurons: list of ints specifying #neurons in each hidden layer
    :param learning_rate: float, learning rate
    :param epochs: number of training epochs
    :return: list of outputs for each of the 4 possible inputs
    """
    # Prepare data
    X, y = get_boolean_data(func_name)

    # Activation function choice
    if activation == 0:
        activation_str = 'sigmoid'
    else:
        activation_str = 'tanh'

    # Build NN
    nn = NeuralNetwork(
        input_size=2,
        hidden_layers=hidden_layers,
        hidden_neurons=hidden_neurons,
        output_size=1,
        activation=activation_str,
        learning_rate=learning_rate,
        epochs=epochs
    )

    # Train
    nn.fit(X, y)

    # Test
    predictions = nn.predict(X)
    return predictions

def run_demo(boolean_func, activation_flag, hidden_layers, hidden_neurons):
    """
    Trains the NN on user-specified boolean functions (AND, OR, XOR or ALL) and prints the results.
    :param boolean_func: 'AND', 'OR', 'XOR' or 'ALL'
    :param activation_flag: 0 for sigmoid, 1 for tanh
    :param hidden_layers: number of hidden layers (int)
    :param hidden_neurons: list of ints specifying #neurons in each hidden layer
    """
    if boolean_func == 'ALL':
        funcs = ['AND', 'OR', 'XOR']
    else:
        funcs = [boolean_func]

    for f in funcs:
        preds = train_and_test_boolean_function(
            func_name=f,
            activation=activation_flag,
            hidden_layers=hidden_layers,
            hidden_neurons=hidden_neurons,
            learning_rate=0.1,
            epochs=10000
        )
        # Print results
        print(f"\n{f}:")
        X = [[0,0],[0,1],[1,0],[1,1]]
        for i, inp in enumerate(X):
            print(f"{tuple(inp)} -> {preds[i]:.4f}")

# Example usage (uncomment the lines below to run in Jupyter):
# run_demo(
#    boolean_func='ALL',  # could be 'AND', 'OR', 'XOR', or 'ALL'
#    activation_flag=0,   # 0 for sigmoid, 1 for tanh
#    hidden_layers=1,     # number of hidden layers
#    hidden_neurons=[4]   # list of neuron counts for each hidden layer
#)

print("Neural Network code loaded. You can call run_demo(...) to train/test.")
print("\nSignificance of 'bias' neuron: Each layer in the network (except for the input layer) typically has an additional ")
print("parameter that acts as a constant offset to help the activations shift. This is implemented as an extra column in ")
print("the weight matrix, effectively a 'bias neuron'. It does not count towards the user-specified number of neurons, but ")
print("it is always added internally for each layer.")
